
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 강의 - 오프라인 vs. 온라인 평가 전략

## 개요

효과적인 에이전트 평가를 위해서는 정제된 데이터셋을 활용한 오프라인 검증과 실제 운영 환경에서의 온라인 모니터링이 모두 필요합니다. 본 강의에서는 오프라인과 온라인 평가의 상호 보완적인 역할, 각각의 장단점, 그리고 두 평가 방식 간의 피드백 루프를 구축하는 방법을 살펴봅니다.

각 접근 방식을 언제 적용해야 하는지 검토하고, 구체적인 구현 패턴을 이해하며, 엄격한 품질 표준을 유지하면서도 실제 사용 환경에 맞춰 진화하는 평가 시스템을 구축하는 방법을 배웁니다.

**학습 목표**

본 강의를 마치면 여러분은 다음을 수행할 수 있습니다.
- 오프라인과 온라인 평가 전략을 언제 사용할지 파악
- 각 접근법의 강점과 한계를 이해
- 오프라인과 온라인 평가가 어떻게 상호 보완하는지 설명
- 운영 환경과 평가 환경 간의 피드백 루프를 구축하는 방법 설명
- 종합적인 평가 전략을 위한 최적의 실무 지침 파악

## A. 평가 전략 개요

### A1. 두 가지 상호 보완적 접근 방식

![offline-vs-online-evaluation.png](../Includes/images/Evaluation with MLflow/offline-vs-online-evaluation.png "offline-vs-online-evaluation.png")

에이전트 평가에는 통제된 검증과 실제 환경에서의 모니터링이 모두 필요합니다. 오프라인 평가와 온라인 평가는 개발 및 배포 수명 주기 전반에 걸쳐 에이전트의 품질을 보장하기 위해 서로 다르지만 상호 보완적인 목적을 수행합니다.

## B. 오프라인 평가: 배포 전 검증

### B1. 통제된 환경에서의 테스트

오프라인 평가는 에이전트를 배포하기 전에 엄선된 데이터셋을 사용해 테스트하는 방식입니다. 이 방식은 에이전트의 품질을 통제되고 재현 가능한 형태로 검증할 수 있게 해줍니다.

**오프라인 평가의 특징:**

- **통제된 환경**: 에이전트가 받는 입력값을 정확하게 제어할 수 있습니다.
- **재현성**: 동일한 데이터셋을 사용하면 일관된 평가 결과를 얻을 수 있습니다.(LLM의 불확정성으로 인한 미세한 차이는 제외)
- **정답(Ground Truth) 활용 가능**: 예상 답변, 사실 관계 또는 전문가가 작성한 가이드라인을 포함할 수 있습니다.
- **포괄적인 검증 범위**: 예외적인 케이스(Edge case), 실패 모드, 다양한 시나리오를 의도적으로 테스트할 수 있습니다.
- **배포 전 관문**: 품질 기준을 충족하지 못하는 에이전트가 배포되는 것을 방지합니다.

### B2. 오프라인 평가 워크플로우

**오프라인 평가 워크플로우:**

1. **데이터셋 생성**: 예상되는 사용 사례, 예외 케이스, 기인된 실패 패턴을 포괄하는 대표적인 예시들을 엄선합니다.
2. **정답 라벨링**: 예상 답변, 사실 관계 또는 가이드라인을 추가합니다.(일반적으로 도메인 전문가가 참여합니다.)
3. **에이전트 실행**: 데이터셋을 바탕으로 에이전트를 구동하여 답변을 생성합니다.
4. **평가 지표(Scorer) 적용**: 복수의 평가 기준을 적용해 다양한 품질 차원을 측정합니다.
5. **분석**: 취합된 지표와 개별 실패 사례를 검토하여 개선 점을 도출합니다.
6. **반복**: 에이전트를 업데이트하고, 다시 평가한 뒤 결과를 비교합니다.

### B3. 오프라인 평가의 장단점

**장점**
- 사용자가 에이전트를 접하기 전에 철저한 검증이 가능합니다.
- 서로 다른 에이전트 설정에 대한 A/B 테스트를 지원합니다.
- 실제 운영 환경 모니터링을 위한 기준 지표(Baseline)를 제공합니다.

**한계**
- 데이터셋이 실제 사용자의 행동을 완벽하게 반영하지 못할 수 있습니다.
- 사용 패턴이 진화함에 따라 고정된 데이터셋은 구식이 될 수 있습니다.
- 대규모 확장 시 또는 다채로운 사용자 층에서만 발생하는 문제는 잡아내기 어렵습니다.

## C. 온라인 평가: 프로덕션 모니터링

### C1. 실제 환경 성능 분석

온라인 평가는 프로덕션 환경에서 실제 사용자 상호작용을 바탕으로 에이전트의 성능을 분석합니다. 이 방식은 실제 사용 패턴, 예상치 못한 입력값, 다양한 사용자층을 대상으로 에이전트가 어떻게 작동하는지 파악합니다.

**온라인 평가의 특징:**

- **실제 데이터**: 실제 사용자 쿼리와 행동을 평가합니다
- **규모 검증**: 에이전트가 프로덕션 부하와 요청 다양성을 어떻게 처리하는지 검증합니다
- **드리프트 감지**: 시간이 지남에 따라 성능이 저하되는 시점을 식별합니다
- **지속적인 모니터링**: 일회성 평가가 아닌, 지속적인 품질 신호를 통해 상태를 파악합니다.
- **사용자 피드백 통합**: 자동 평가와 실제 사용자 만족도를 결합합니다

### C2. 온라인 평가 워크플로

**온라인 평가 워크플로우:**

1. **추론 로깅:** AI Gateway 기반의 추론 테이블 기능이 활성화된 에이전트 배포
2. **자동 트레이싱:** 모든 운영 환경 요청에 대해 추적(Trace)을 생성하고 추론 테이블에 기록
3. **예약 평가:** 최근 운영 트레이스를 대상으로 스코어러(Scorer)를 주기적으로 실행
4. **알림 설정:** 품질 저하 감지를 위한 모니터링 알림 구성
5. **피드백 수집:** 사용자의 '좋아요/싫어요' 반응 또는 명시적 피드백 수집
6. **데이터셋 증강:** 오프라인 평가용으로 프로덕션에서 주목할 만한 사례를 뽑아 반영

### C3. MLflow의 프로덕션 모니터링 기능

**MLflow의 프로덕션 모니터링 기능:**

데이터브릭스(Databricks)는 모델 서빙(Model Serving)을 통해 배포된 에이전트에 대해 내장된 프로덕션 모니터링 기능을 제공합니다.
- **자동 평가 집행**: 프로덕션 트레이스를 대상으로 MLflow 평가 지표를 자동으로 실행합니다.
- **품질 대시보드**: 시간에 따른 평가 지표의 변화를 시각화합니다.
- **알림**: 품질 지표가 기준치 아래로 떨어지면 알림을 보냅니다.
- **트레이스 검토**: 개별 실패 사례나 예외적인 케이스(Edge case)를 심층 분석합니다.

### C4. 온라인 평가의 강점과 한계

**장점:**
- 실제 사용자 경험 및 실환경 성능을 반영합니다.
- 오프라인 테스트에서 예상치 못했던 문제를 탐지합니다.
- 지속적인 개선을 위한 데이터를 제공합니다.

**한계:**
- 문제를 감지하기 전에 사용자가 먼저 실패를 경험할 수 있습니다.
- 원인 분석이 더 어렵습니다. (다양한 입력값으로 인한 혼재 변수 발생)
- 프로덕션 인프라와 모니터링 시스템이 구축되어야 합니다.

## D. 상호 보완적 전략

### D1.접근 방식별 활용 시점

효과적인 에이전트 평가는 오프라인과 온라인 접근 방식을 모두 결합하여 이루어집니다.

**오프라인 평가**
- 배포 전 검증 및 품질 검사(Quality gates)
- 다양한 에이전트 구현 방식 간의 비교
- 에이전트 행동에 대한 특정 가설 검증
- 개발 단계에서의 신속한 반복 개선

**온라인 평가**
- 오프라인 평가 결과가 실제 운영 환경에서도 동일하게 적용되는지 검증
- 성능 저하 또는 모델 드리프트(Drift) 감지
- 실제 사용자의 니즈와 페인 포인트(Pain points) 파악
- 실제 사용 데이터를 기반으로 평가 데이터셋 구축

### D2. 피드백 루프 생성

**피드백 루프 생성**

1. 신중하게 선별된 데이터셋을 바탕으로 오프라인 평가부터 시작합니다.
2. 모니터링 기능을 활성화한 상태로 운영 환경에 배포합니다.
3. 운영 환경의 트레이스(Trace) 데이터를 분석하여 오류 및 예외 상황(Edge cases)을 식별합니다.
4. 실제 운영 예시들을 오프라인 평가 데이터셋에 추가합니다.
5. 고도화된 데이터셋을 활용해 재배포 전 개선 사항을 검증합니다.

이러한 선순환 구조를 통해 철저한 배포 전 검증을 유지하는 동시에, 실제 사용 환경에 대한 이해도를 바탕으로 평가 체계를 지속적으로 발전시킬 수 있습니다.

## E. 종합 평가 데이터셋 구축

### E1. 데이터셋 구성 원칙

평가의 품질은 데이터셋에 결정적으로 좌우됩니다. 잘 설계된 평가 데이터셋은 대표성과 다양성을 갖추어야 하며, 적절한 정답셋(Ground Truth)을 포함해야 합니다.

**데이터셋 구성 원칙**

**1. 대표성**: 일반적인 사용자 쿼리와 사용 사례를 반영한 예시를 포함
- 프로덕션 로그나 사용자 조사를 분석하여 공통 패턴을 식별
- 쿼리 유형의 분포가 예상 사용량과 일치하는지 확인
- 다양한 쿼리 복잡도 수준 포함

**2. 예외 사례 커버리지**: 경계 조건과 잠재적 실패를 의도적으로 시험
- 여러 방식으로 해석될 수 있는 모호한 쿼리
- 검색용 말뭉치에 없는 정보가 필요한 쿼리
- 실패를 유발하도록 설계된 적대적 예시(Adversarial Examples)
- 예상치 못한 형식이나 표현으로 된 쿼리


**3. 다양성 차원**: 입력값의 여러 측면을 다양화
- 쿼리 길이 (간결한 vs. 장황함)
- 쿼리 복잡성 (단순 사실 조회 vs. 다단계 추론)
- 도메인 커버리지(에이전트가 다뤄야 할 다양한 주제)
- 사용자 전문성 수준 (초보자 vs. 전문가 도메인 지식)

### E2. 다양성 및 정답셋(Ground Truth) 접근 방식


**정답셋(Ground Truth) 접근 방식**

**전문가 라벨링**: 도메인 전문가가 예상 출력 또는 팩트 세트를 생성합니다.
- 가장 정확하지만 리소스 집약적입니다.
- 중요도가 높은 애플리케이션에 필수적입니다.
- 재사용 가능한 골드 스탠다드 데이터 세트를 생성합니다.

**합성 데이터 생성**: LLM을 사용하여 예상 응답을 생성합니다.
- 확장 가능하지만 검증이 필요합니다.
- 초기 평가 데이터셋을 빠르게 구축(Bootstrapping)할 때 유용합니다.
- 사람이 직접 무작위로 샘플을 점검(Spot-check)해야 합니다.

**운영 데이터 마이닝**: 추론 테이블에서 예제를 추출합니다.
- 실제 사용 패턴을 반영합니다.
- 필터링 또는 데이터 정제가 필요할 수 있습니다.
- 새로운 오류 모드를 식별할 수 있습니다.

### E3. 데이터셋 유지 관리

**데이터셋 유지보수**

평가 데이터셋은 지속적인 큐레이션이 필요합니다.
- 실제 운영 환경에서 가져온 새로운 예제를 정기적으로 추가합니다.
- 오래된 예제를 삭제하거나 업데이트합니다.
- 사용 패턴 변화에 따라 카테고리 간 균형을 재조정합니다.
- 에이전트 버전과 함께 데이터셋도 버전 관리합니다.

## F. Unity Catalog와의 통합

### F1. 평가를 위한 거버넌스 계층

Unity Catalog는 평가 워크플로를 위한 거버넌스 레이어를 제공하여 추적 가능성, 액세스 제어 및 데이터 계보(Lineage) 추적을 보장합니다.

**에이전트 등록**: Unity Catalog에 에이전트를 등록하면 다음과 같은 이점이 있습니다.
- 모든 에이전트 종속성(벡터 인덱스, UC 함수, 모델 엔드포인트)이 추적됩니다.
- 액세스 제어를 통해 에이전트를 평가하거나 수정할 수 있는 권한을 결정합니다.
- 데이터 계보를 통해 어떤 평가 데이터 세트와 지표가 각 버전을 생성했는지 보여줍니다.
- 별칭(예: @champion)을 사용하여 깔끔한 배포(Promotion) 워크플로를 구현할 수 있습니다.

### F2. 데이터셋 및 트레이스 저장

**평가 데이터셋 저장소**: 데이터 세트를 Delta 테이블이나 볼륨으로 저장합니다.
- 에이전트 버전과 함께 데이터 세트 버전을 관리합니다.
- 민감한 평가 데이터에 액세스 제어를 적용합니다.
- SQL 기반의 데이터 세트 분석 및 강화를 지원합니다.
- 공동 작업을 통한 데이터 세트 관리를 지원합니다.

**트레이스 저장**: MLflow 트레이스(Trace)는 Unity Catalog에 저장됩니다.
- 분석을 위해 SQL을 사용해 트레이스를 쿼리할 수 있습니다.
- 트레이스를 평가 결과와 결합하여 더 심층적인 인사이트를 얻을 수 있습니다.
- 운영 환경의 트레이스 데이터에 데이터 거버넌스 정책을 적용합니다.
- 팀 간에 적절한 수준의 트레이스 공유를 지원합니다.

이러한 통합을 통해 평가 인프라는 데이터 플랫폼의 다른 요소와 동일한 거버넌스, 보안 및 협업 기능을 활용할 수 있습니다.

## G. 핵심 요약

효과적인 에이전트 평가를 위해서는 오프라인과 온라인 전략이 모두 필요합니다.

1. **상호 보완적 접근법**: 오프라인 평가는 통제된 검증을 제공하고, 온라인 평가는 실제 성능을 포착합니다
2. **피드백 루프**: 프로덕션 데이터가 오프라인 평가 데이터셋을 보강해 지속적인 개선 주기를 만듭니다
3. **데이터셋 품질**: 대표성 있고 다양하며 잘 관리된 데이터셋은 의미 있는 평가에 필수적입니다
4. **거버넌스 통합**: Unity Catalog는 확장 가능하고 거버넌스가 있는 평가 워크플로우를 위한 인프라를 제공합니다
5. **지속적인 진화**: 평가 전략은 에이전트, 사용 패턴, 요구사항이 진화함에 따라 조정되어야 합니다

철저한 오프라인 검증과 포괄적인 온라인 모니터링의 결합은 개발 및 배포 수명 주기 전반에서 에이전트의 품질을 보장하는 견고한 평가 프레임워크를 구축합니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>